In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_procurement.csv")
df.head()

,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,Compliance,Delivery_Missing,Delivery_Delay_Days,Savings_Per_Unit,Total_Savings,Defect_Rate
0,PO-00001,ALPHA_INC,2023-10-17,2023-10-25,OFFICE SUPPLIES,CANCELLED,1176,20.13,17.81,0.0,YES,False,8.0,2.32,2728.32,0.000000
1,PO-00002,DELTA_LOGISTICS,2022-04-25,2022-05-05,OFFICE SUPPLIES,DELIVERED,1509,39.32,37.34,235.0,YES,False,10.0,1.98,2987.82,0.155732
2,PO-00003,GAMMA_CO,2022-01-26,2022-02-15,MRO,DELIVERED,910,95.51,92.26,41.0,YES,False,20.0,3.25,2957.50,0.045055
3,PO-00004,BETA_SUPPLIES,2022-10-09,2022-10-28,PACKAGING,DELIVERED,1344,99.85,95.52,112.0,YES,False,19.0,4.33,5819.52,0.083333
4,PO-00005,DELTA_LOGISTICS,2022-09-08,2022-09-20,RAW MATERIALS,DELIVERED,1180,64.07,60.53,171.0,NO,False,12.0,3.54,4177.20,0.144915


In [3]:
df.columns = df.columns.str.strip()
df.columns

Index(['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category',
       'Order_Status', 'Quantity', 'Unit_Price', 'Negotiated_Price',
       'Defective_Units', 'Compliance', 'Delivery_Missing',
       'Delivery_Delay_Days', 'Savings_Per_Unit', 'Total_Savings',
       'Defect_Rate'],
      dtype='object')

In [4]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'], errors='coerce')

In [5]:
df['Total_Cost'] = df['Quantity'] * df['Unit_Price']
df['Savings'] = df['Quantity'] * (df['Unit_Price'] - df['Negotiated_Price'])

In [6]:
supplier_spend = df.groupby('Supplier').agg(
    Total_Spend=('Total_Cost', 'sum'),
    Total_Savings=('Savings', 'sum'),
    Orders=('PO_ID', 'count')
).reset_index()

supplier_spend = supplier_spend.sort_values('Total_Spend', ascending=False)

supplier_spend.head(10)

,Supplier,Total_Spend,Total_Savings,Orders
1,BETA_SUPPLIES,10748606.79,889940.89,156
3,EPSILON_GROUP,10696136.24,844980.18,166
2,DELTA_LOGISTICS,10018216.96,781976.49,171
4,GAMMA_CO,9313230.13,725308.42,143
0,ALPHA_INC,8528632.74,688920.49,141


In [7]:
df['Month'] = df['Order_Date'].dt.to_period('M').astype(str)

category_trend = df.groupby(['Item_Category', 'Month']).agg(
    Spend=('Total_Cost', 'sum')
).reset_index()

category_trend.head()

,Item_Category,Month,Spend
0,ELECTRONICS,2022-01,502273.33
1,ELECTRONICS,2022-02,215678.37
2,ELECTRONICS,2022-03,261164.19
3,ELECTRONICS,2022-04,793598.98
4,ELECTRONICS,2022-05,258032.20


In [8]:
category_trend['Spend_Lag'] = category_trend.groupby('Item_Category')['Spend'].shift(1)
category_trend['Growth_Rate'] = (category_trend['Spend'] - category_trend['Spend_Lag']) / category_trend['Spend_Lag']

category_trend.head()

,Item_Category,Month,Spend,Spend_Lag,Growth_Rate
0,ELECTRONICS,2022-01,502273.33,NaN,NaN
1,ELECTRONICS,2022-02,215678.37,502273.33,-0.570596
2,ELECTRONICS,2022-03,261164.19,215678.37,0.210897
3,ELECTRONICS,2022-04,793598.98,261164.19,2.038698
4,ELECTRONICS,2022-05,258032.20,793598.98,-0.674858


In [9]:
df['Price_Reduction'] = df['Unit_Price'] - df['Negotiated_Price']
df['Reduction_Rate'] = df['Price_Reduction'] / df['Unit_Price']

negotiation_analysis = df.groupby('Supplier').agg(
    Avg_Reduction=('Reduction_Rate', 'mean'),
    Total_Savings=('Savings', 'sum')
).reset_index()

negotiation_analysis.sort_values('Avg_Reduction', ascending=False).head(10)

,Supplier,Avg_Reduction,Total_Savings
0,ALPHA_INC,0.082105,688920.49
3,EPSILON_GROUP,0.080405,844980.18
4,GAMMA_CO,0.079849,725308.42
1,BETA_SUPPLIES,0.078277,889940.89
2,DELTA_LOGISTICS,0.078120,781976.49


In [10]:
delay_analysis = df.groupby('Supplier').agg(
    Avg_Delay=('Delivery_Delay_Days', 'mean'),
    Max_Delay=('Delivery_Delay_Days', 'max'),
    Orders=('PO_ID', 'count')
).reset_index()

delay_analysis.sort_values('Avg_Delay', ascending=False).head(10)

,Supplier,Avg_Delay,Max_Delay,Orders
1,BETA_SUPPLIES,11.272727,20.0,156
3,EPSILON_GROUP,10.865772,20.0,166
2,DELTA_LOGISTICS,10.854305,20.0,171
0,ALPHA_INC,10.606838,20.0,141
4,GAMMA_CO,10.192308,20.0,143


In [11]:
defect_analysis = df.groupby('Supplier').agg(
    Avg_Defect_Rate=('Defect_Rate', 'mean'),
    Total_Defects=('Defective_Units', 'sum')
).reset_index()

defect_analysis.sort_values('Avg_Defect_Rate', ascending=False).head(10)

,Supplier,Avg_Defect_Rate,Total_Defects
2,DELTA_LOGISTICS,0.108666,19678.0
1,BETA_SUPPLIES,0.082723,13838.0
4,GAMMA_CO,0.044975,7034.0
3,EPSILON_GROUP,0.026063,4682.0
0,ALPHA_INC,0.018905,2717.0


In [12]:
supplier_score = supplier_spend.merge(delay_analysis, on='Supplier').merge(defect_analysis, on='Supplier')

supplier_score['Risk_Score'] = (
    supplier_score['Avg_Delay'] +
    supplier_score['Avg_Defect_Rate'] * 100
)

supplier_score.sort_values('Risk_Score', ascending=False).head(10)

,Supplier,Total_Spend,Total_Savings,Orders_x,Avg_Delay,Max_Delay,Orders_y,Avg_Defect_Rate,Total_Defects,Risk_Score
2,DELTA_LOGISTICS,10018216.96,781976.49,171,10.854305,20.0,171,0.108666,19678.0,21.720879
0,BETA_SUPPLIES,10748606.79,889940.89,156,11.272727,20.0,156,0.082723,13838.0,19.545054
3,GAMMA_CO,9313230.13,725308.42,143,10.192308,20.0,143,0.044975,7034.0,14.689844
1,EPSILON_GROUP,10696136.24,844980.18,166,10.865772,20.0,166,0.026063,4682.0,13.472041
4,ALPHA_INC,8528632.74,688920.49,141,10.606838,20.0,141,0.018905,2717.0,12.497386


In [13]:
duplicates = df[df.duplicated(subset=['PO_ID', 'Supplier', 'Order_Date'], keep=False)]

duplicates.head()

,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,...,Delivery_Missing,Delivery_Delay_Days,Savings_Per_Unit,Total_Savings,Defect_Rate,Total_Cost,Savings,Month,Price_Reduction,Reduction_Rate


In [14]:
category_summary = df.groupby('Item_Category').agg(
    Total_Spend=('Total_Cost', 'sum'),
    Avg_Defect=('Defect_Rate', 'mean'),
    Avg_Delay=('Delivery_Delay_Days', 'mean')
).reset_index()

category_summary.sort_values('Total_Spend', ascending=False)

,Item_Category,Total_Spend,Avg_Defect,Avg_Delay
1,MRO,11028921.17,0.053427,11.597315
2,OFFICE SUPPLIES,10851679.19,0.063922,10.406667
0,ELECTRONICS,9385426.83,0.058092,11.021583
4,RAW MATERIALS,9180338.67,0.062721,10.341880
3,PACKAGING,8858457.00,0.050525,10.407407


In [15]:
supplier_spend.to_csv("supplier_spend.csv", index=False)
category_summary.to_csv("category_summary.csv", index=False)
delay_analysis.to_csv("delay_analysis.csv", index=False)
defect_analysis.to_csv("defect_analysis.csv", index=False)
supplier_score.to_csv("supplier_risk_score.csv", index=False)